In [ ]:
# Cell 1: Load and prepare GSE data
import pandas as pd
import numpy as np
import os
import pickle
from datetime import datetime
from biolearn.data_library import DataLibrary

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def load_gse_data(accession: str):
    """
    Load GSE data using biolearn and save locally as CSV files.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
    
    Returns:
        tuple: (dnam_df, metadata_df) - methylation data and metadata as pandas DataFrames
    """
    _log(f"Starting {accession} data loading...")
    
    # Create local data directory
    data_dir = "2_poc_simulacra"
    os.makedirs(data_dir, exist_ok=True)
    
    # Check if data already exists
    dnam_path = os.path.join(data_dir, f"{accession}_dnam.csv")
    metadata_path = os.path.join(data_dir, f"{accession}_metadata.csv")
    
    if os.path.exists(dnam_path) and os.path.exists(metadata_path):
        _log(f"Loading existing {accession} data from local files...")
        dnam_df = pd.read_csv(dnam_path, index_col=0)
        metadata_df = pd.read_csv(metadata_path, index_col=0)
        _log(f"Loaded {accession}_dnam.csv: {dnam_df.shape[0]} rows, {dnam_df.shape[1]} columns")
        _log(f"Loaded {accession}_metadata.csv: {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns")
        return dnam_df, metadata_df
    
    _log(f"Downloading {accession} data from biolearn (LONG OPERATION)...")
    start_time = datetime.now()
    
    # Load data using biolearn
    library = DataLibrary()
    data = library.get(accession)
    if data is None:
        raise ValueError(f"{accession} dataset not found in biolearn library")
    
    data = data.load()
    
    # Extract methylation data (transpose to have samples as rows)
    dnam_df = data.dnam.T
    metadata_df = data.metadata.copy()
    
    # Ensure index alignment
    common_samples = dnam_df.index.intersection(metadata_df.index)
    dnam_df = dnam_df.loc[common_samples]
    metadata_df = metadata_df.loc[common_samples]
    
    download_duration = datetime.now() - start_time
    _log(f"Data download completed in {download_duration.total_seconds():.2f} seconds")
    
    _log(f"Saving {accession} data to local CSV files (LONG OPERATION)...")
    save_start = datetime.now()
    
    # Save methylation data
    dnam_df.to_csv(dnam_path)
    _log(f"Saved {accession}_dnam.csv: {dnam_df.shape[0]} rows, {dnam_df.shape[1]} columns")
    
    # Save metadata
    metadata_df.to_csv(metadata_path)
    _log(f"Saved {accession}_metadata.csv: {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns")
    
    save_duration = datetime.now() - save_start
    _log(f"Data saving completed in {save_duration.total_seconds():.2f} seconds")
    
    return dnam_df, metadata_df

# Execute the function for GSE42861
dnam_df, metadata_df = load_gse_data('GSE42861')

_log("GSE42861 data loading completed successfully!")
_log(f"Final dnam shape: {dnam_df.shape}")
_log(f"Final metadata shape: {metadata_df.shape}")
_log(f"Metadata columns: {list(metadata_df.columns)}")


[2025-10-12 12:30:34] Starting GSE42861 data loading...
[2025-10-12 12:30:34] Downloading GSE42861 data from biolearn (LONG OPERATION)...
[2025-10-12 12:30:49] Data download completed in 14.56 seconds
[2025-10-12 12:30:49] Saving data to local CSV files (LONG OPERATION)...
[2025-10-12 12:48:22] Saved dnam.csv: 689 rows, 485577 columns
[2025-10-12 12:48:22] Saved metadata.csv: 689 rows, 4 columns
[2025-10-12 12:48:22] Data saving completed in 1052.84 seconds
[2025-10-12 12:48:22] GSE42861 data loading completed successfully!
[2025-10-12 12:48:22] Final dnam shape: (689, 485577)
[2025-10-12 12:48:22] Final metadata shape: (689, 4)
[2025-10-12 12:48:22] Metadata columns: ['disease', 'age', 'sex', 'smoking']


In [ ]:
# Cell 2: Generate embeddings with Pythae
import os
import sys
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
from pythae.models import VAE, VAEConfig
from pythae.trainers import BaseTrainerConfig
from pythae.pipelines.training import TrainingPipeline
import glob

# Import our custom dataset classes
from iterable_csv_dataset import PythaeIterableDataset, DNAmDataset

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def find_model_path(model_dir):
    """Find the actual model path within the timestamped subdirectory."""
    if not os.path.exists(model_dir):
        return None
    
    # Look for VAE_training_* directories
    training_dirs = glob.glob(os.path.join(model_dir, "VAE_training_*"))
    if not training_dirs:
        return None
    
    # Get the most recent training directory
    latest_training_dir = max(training_dirs, key=os.path.getctime)
    final_model_path = os.path.join(latest_training_dir, "final_model")
    
    if os.path.exists(final_model_path):
        return final_model_path
    return None

def generate_embeddings_with_pythae(accession: str, n_dnam_cols: int = 100, latent_dim: int = 512, epochs: int = 20):
    """
    Generate embeddings using Pythae VAE on GSE data.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
        n_dnam_cols (int): Number of DNAm columns to use for training
        latent_dim (int): Latent dimension for VAE
        epochs (int): Number of training epochs
    
    Returns:
        str: Path to the generated embeddings CSV file
    """
    _log(f"Starting Pythae embedding generation for {accession}...")
    
    # Create local data directory
    data_dir = "2_poc_simulacra"
    os.makedirs(data_dir, exist_ok=True)
    
    # Define paths - use the actual file names that exist
    embeddings_path = os.path.join(data_dir, f"{accession}_embeddings.csv")
    model_dir = os.path.join(data_dir, f"{accession}_vae_model")
    dnam_path = os.path.join(data_dir, "dnam.csv")
    metadata_path = os.path.join(data_dir, "metadata.csv")
    
    # Check if embeddings already exist
    if os.path.exists(embeddings_path):
        _log(f"Loading existing embeddings from {embeddings_path}...")
        embeddings_df = pd.read_csv(embeddings_path, index_col=0)
        _log(f"Loaded embeddings: {embeddings_df.shape[0]} rows, {embeddings_df.shape[1]} columns")
        return embeddings_path
    
    # Check if model already exists
    model_path = find_model_path(model_dir)
    if model_path:
        _log(f"Loading existing VAE model from {model_path}...")
        model = VAE.load_from_folder(model_path)
        _log("VAE model loaded successfully")
    else:
        _log("Training new VAE model (LONG OPERATION)...")
        train_start = datetime.now()
        
        # Create train/validation split
        ds = {}
        ds['train'], ds['val'] = DNAmDataset.split_dataset(
            split_points=0.8,
            dnam_path=dnam_path,
            metadata_path=metadata_path,
            n_dnam_cols=n_dnam_cols,
            target_metadata_col="disease",
            target_metadata_type="categorical",
            shuffle=True,
            seed=42,
            on_unmatched='warn',
            join_on='Unnamed: 0',
        )
        
        _log(f"Dataset split created - Train: {len(ds['train'])}, Val: {len(ds['val'])}")
        
        # Create VAE model
        model_config = VAEConfig(
            input_dim=(n_dnam_cols,),
            latent_dim=latent_dim
        )
        
        # Create training config
        training_config = BaseTrainerConfig(
            output_dir=model_dir,
            learning_rate=1e-3,
            per_device_train_batch_size=512,
            per_device_eval_batch_size=512,
            num_epochs=epochs,
            keep_best_on_train=True
        )
        
        # Create training pipeline
        pipeline = TrainingPipeline(
            model=VAE(model_config=model_config),
            training_config=training_config
        )
        
        # Train the model
        pipeline(
            train_data=ds['train'],
            eval_data=ds['val']
        )
        
        train_duration = datetime.now() - train_start
        _log(f"VAE training completed in {train_duration.total_seconds():.2f} seconds")
        
        # Find and load the trained model
        model_path = find_model_path(model_dir)
        if model_path:
            model = VAE.load_from_folder(model_path)
            _log("VAE model loaded successfully")
        else:
            raise FileNotFoundError(f"Could not find trained model in {model_dir}")
    
    # Generate embeddings for all data
    _log("Generating embeddings for all data (LONG OPERATION)...")
    embed_start = datetime.now()
    
    # Create dataset for all data
    all_ds = PythaeIterableDataset(
        dnam_path=dnam_path,
        metadata_path=metadata_path,
        n_dnam_cols=n_dnam_cols,
        target_metadata_col="disease",
        target_metadata_type="categorical",
        join_on="Unnamed: 0",
    )
    
    # Get the list of indices from metadata
    index_list = list(all_ds.metadata.index)
    
    # Generate embeddings
    model.eval()
    buffer = []
    with torch.no_grad():
        with open(embeddings_path, 'w') as fout:
            header = ','.join([f'emb_{i}' for i in range(model.latent_dim)])
            fout.write(f'id,{header}\n')
            
            for idx, (X, _) in tqdm(enumerate(all_ds), desc="Generating embeddings"):
                X_tensor = X.to(dtype=torch.float32).unsqueeze(0)
                embedding = model.encoder(X_tensor)['embedding'].cpu().numpy().squeeze()
                
                # Use the original index from metadata
                row_id = index_list[idx] if idx < len(index_list) else idx
                row = ','.join([str(x) for x in embedding])
                buffer.append(f'{row_id},{row}\n')
                
                # Write buffer periodically
                if len(buffer) >= 64:
                    fout.writelines(buffer)
                    buffer.clear()
            
            # Write remaining buffer
            if buffer:
                fout.writelines(buffer)
    
    embed_duration = datetime.now() - embed_start
    _log(f"Embedding generation completed in {embed_duration.total_seconds():.2f} seconds")
    
    # Load and display the embeddings
    embeddings_df = pd.read_csv(embeddings_path, index_col=0)
    _log(f"Generated embeddings: {embeddings_df.shape[0]} rows, {embeddings_df.shape[1]} columns")
    
    return embeddings_path

# Execute the function for GSE42861
embeddings_path = generate_embeddings_with_pythae('GSE42861', n_dnam_cols=100, latent_dim=512, epochs=20)

_log("Pythae embedding generation completed successfully!")
_log(f"Embeddings saved to: {embeddings_path}")

# Display first few rows of embeddings
embeddings_df = pd.read_csv(embeddings_path, index_col=0)
_log(f"Final embeddings shape: {embeddings_df.shape}")
_log("First 5 rows of embeddings:")
display(embeddings_df)


[2025-10-12 21:47:43] Starting Pythae embedding generation for GSE42861...
[2025-10-12 21:47:43] Loading existing VAE model from 2_poc_simulacra\GSE42861_vae_model\VAE_training_2025-10-12_20-48-52\final_model...
[2025-10-12 21:47:44] VAE model loaded successfully
[2025-10-12 21:47:44] Generating embeddings for all data (LONG OPERATION)...


Generating embeddings: 689it [02:00,  5.74it/s] 


[2025-10-12 21:49:44] Embedding generation completed in 120.31 seconds
[2025-10-12 21:49:44] Generated embeddings: 689 rows, 512 columns
[2025-10-12 21:49:44] Pythae embedding generation completed successfully!
[2025-10-12 21:49:44] Embeddings saved to: 2_poc_simulacra\GSE42861_embeddings.csv
[2025-10-12 21:49:44] Final embeddings shape: (689, 512)
[2025-10-12 21:49:44] First 5 rows of embeddings:
               emb_0     emb_1     emb_2     emb_3     emb_4     emb_5  \
id                                                                       
GSM1051525  0.000379  0.007248 -0.011952 -0.006601 -0.002977  0.001650   
GSM1051526 -0.002986  0.003322 -0.002674 -0.010356  0.005399  0.009582   
GSM1051527 -0.006568  0.003980  0.000052 -0.007087 -0.002809  0.004245   
GSM1051528  0.000215  0.009518 -0.009776 -0.007672  0.004302  0.010845   
GSM1051529 -0.001178  0.006284 -0.006167 -0.007700  0.001529  0.008875   

               emb_6     emb_7     emb_8     emb_9  ...   emb_502   emb_503  \
i

In [9]:
# Cell 3: Combine embeddings with target column
import pandas as pd
import numpy as np
from datetime import datetime

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def create_embeddings_with_target(accession: str, target_col: str = "disease"):
    """
    Create a new dataframe combining embeddings with target column.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
        target_col (str): Name of the target column from metadata
    
    Returns:
        pd.DataFrame: Combined dataframe with target as first column, followed by embeddings
    """
    _log(f"Creating combined dataframe for {accession} with target column '{target_col}'...")
    
    # Define paths
    data_dir = "2_poc_simulacra"
    embeddings_path = os.path.join(data_dir, f"{accession}_embeddings.csv")
    metadata_path = os.path.join(data_dir, "metadata.csv")
    combined_path = os.path.join(data_dir, f"{accession}_embeddings_with_target.csv")
    
    # Check if combined file already exists
    if os.path.exists(combined_path):
        _log(f"Loading existing combined dataframe from {combined_path}...")
        combined_df = pd.read_csv(combined_path, index_col=0)
        _log(f"Loaded combined dataframe: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")
        return combined_df
    
    # Load embeddings and metadata
    _log("Loading embeddings and metadata...")
    embeddings_df = pd.read_csv(embeddings_path, index_col=0)
    metadata_df = pd.read_csv(metadata_path, index_col=0)
    
    _log(f"Embeddings shape: {embeddings_df.shape}")
    _log(f"Metadata shape: {metadata_df.shape}")
    
    # Check if target column exists in metadata
    if target_col not in metadata_df.columns:
        available_cols = list(metadata_df.columns)
        raise ValueError(f"Target column '{target_col}' not found in metadata. Available columns: {available_cols}")
    
    # Join on index to combine the dataframes
    _log("Combining embeddings with target column...")
    combined_df = pd.concat([metadata_df[[target_col]], embeddings_df], axis=1, join='inner')
    
    _log(f"Combined dataframe shape: {combined_df.shape}")
    _log(f"Target column '{target_col}' unique values: {combined_df[target_col].unique()}")
    
    # Save the combined dataframe
    _log(f"Saving combined dataframe to {combined_path}...")
    combined_df.to_csv(combined_path)
    _log("Combined dataframe saved successfully")
    
    return combined_df

# Execute the function for GSE42861
combined_df = create_embeddings_with_target('GSE42861', target_col='disease')

_log("Combined dataframe creation completed successfully!")
_log(f"Final combined dataframe shape: {combined_df.shape}")
_log(f"Columns: {list(combined_df.columns)}")
_log("First 5 rows:")
display(combined_df)


[2025-10-12 22:03:27] Creating combined dataframe for GSE42861 with target column 'disease'...
[2025-10-12 22:03:27] Loading embeddings and metadata...
[2025-10-12 22:03:27] Embeddings shape: (689, 512)
[2025-10-12 22:03:27] Metadata shape: (689, 4)
[2025-10-12 22:03:27] Combining embeddings with target column...
[2025-10-12 22:03:27] Combined dataframe shape: (689, 513)
[2025-10-12 22:03:27] Target column 'disease' unique values: ['rheumatoid arthritis' 'Normal']
[2025-10-12 22:03:27] Saving combined dataframe to 2_poc_simulacra\GSE42861_embeddings_with_target.csv...
[2025-10-12 22:03:28] Combined dataframe saved successfully
[2025-10-12 22:03:28] Combined dataframe creation completed successfully!
[2025-10-12 22:03:28] Final combined dataframe shape: (689, 513)
[2025-10-12 22:03:28] Columns: ['disease', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'em

,disease,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_502,emb_503,emb_504,emb_505,emb_506,emb_507,emb_508,emb_509,emb_510,emb_511
GSM1051525,rheumatoid arthritis,0.000379,0.007248,-0.011952,-0.006601,-0.002977,0.001650,-0.002853,-0.006493,-0.003603,...,0.001855,0.004305,0.002511,0.002612,0.007929,-0.006192,-0.000926,-0.000606,0.011051,0.003112
GSM1051526,rheumatoid arthritis,-0.002986,0.003322,-0.002674,-0.010356,0.005399,0.009582,-0.007116,-0.010195,0.004752,...,0.002104,0.006323,0.003349,0.000850,0.008739,-0.000499,-0.003584,-0.003576,0.008540,-0.003069
GSM1051527,rheumatoid arthritis,-0.006568,0.003980,0.000052,-0.007087,-0.002809,0.004245,-0.003250,-0.004742,0.003934,...,0.005058,0.002357,0.002605,0.004585,0.003063,-0.010164,-0.005583,-0.003053,0.012815,0.001405
GSM1051528,rheumatoid arthritis,0.000215,0.009518,-0.009776,-0.007672,0.004302,0.010845,-0.001150,-0.007701,0.006775,...,0.004212,0.011965,0.009021,-0.004224,0.006240,0.002095,0.002774,-0.006467,-0.000980,-0.003478
GSM1051529,rheumatoid arthritis,-0.001178,0.006284,-0.006167,-0.007700,0.001529,0.008875,-0.003372,-0.004588,0.002095,...,0.004174,0.004763,0.007382,0.001549,0.006101,-0.000170,0.001114,-0.002347,0.005288,-0.000686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1052209,Normal,-0.003765,0.008041,-0.006019,-0.009293,0.003222,0.009334,-0.003292,-0.012899,0.001117,...,0.002203,0.008307,0.006887,-0.001503,0.007390,0.002973,-0.002571,-0.001365,0.000916,0.001138
GSM1052210,Normal,-0.004615,-0.005707,-0.013385,0.008344,-0.020104,0.005249,-0.007903,0.005094,-0.011255,...,-0.016663,-0.014000,-0.005464,-0.007783,0.001713,-0.010014,-0.012169,0.010438,0.003982,0.016364
GSM1052211,Normal,-0.005845,0.007845,-0.008336,-0.003659,0.002398,0.004383,-0.005194,-0.007207,0.000569,...,0.000923,0.008879,0.010149,0.002801,0.009749,-0.000687,-0.004634,-0.000321,0.003844,0.001026
GSM1052212,Normal,-0.005973,0.000146,-0.000831,-0.002002,-0.001332,0.005437,-0.010735,-0.005272,-0.003169,...,0.004573,0.003191,0.005803,0.006951,0.006752,-0.002935,-0.005506,0.006975,0.013059,0.009616


In [1]:
# Cell 4: Benchmark Ridge Classifiers with Different Training Sets
import pandas as pd
import numpy as np
import os
import pickle
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sdv.single_table import GaussianCopulaSynthesizer, CTGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def train_ridge_classifier_fair(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    random_state: int = 42
) -> tuple:
    """Train Ridge classifier on given train/test splits and return model + metrics."""
    
    # Handle any NaN values
    train_mask = (~X_train.isna().any(axis=1)) & y_train.notna()
    test_mask = (~X_test.isna().any(axis=1)) & y_test.notna()
    
    X_train_clean = X_train.loc[train_mask]
    y_train_clean = y_train.loc[train_mask]
    X_test_clean = X_test.loc[test_mask]
    y_test_clean = y_test.loc[test_mask]
    
    # Train model
    model = Pipeline([
        ('scaler', StandardScaler(with_mean=False)),
        ('ridge', RidgeClassifier(random_state=random_state))
    ])
    model.fit(X_train_clean, y_train_clean)
    
    # Evaluate
    y_pred = model.predict(X_test_clean)
    metrics = {
        'accuracy': accuracy_score(y_test_clean, y_pred),
        'f1_macro': f1_score(y_test_clean, y_pred, average='macro'),
        'report': classification_report(y_test_clean, y_pred, output_dict=False),
        'y_true': y_test_clean,
        'y_pred': y_pred,
        'test_size': len(y_test_clean)
    }
    
    return model, metrics

def benchmark_classifiers(accession: str, test_fraction: float = 0.2, seed: int = 42, multipliers = (1, 2)):
    """
    Benchmark Ridge classifiers on different training sets with multiple augmentation multipliers.
    
    Args:
        accession (str): GSE accession number
        test_fraction (float): Fraction of data to use for testing (default: 0.2)
        seed (int): Random seed for reproducibility
        multipliers: Either a tuple of augmentation multipliers or a single number
                    Single numbers will be cast to tuple. None is not allowed in multipliers.
                    Examples: (1, 2, 5) or 2 (becomes (2,))
    
    Returns:
        dict: Results for each training set configuration
    """
    # Handle single number multipliers
    if isinstance(multipliers, (int, float)):
        multipliers = (multipliers,)
    
    # Validate multipliers - None is not allowed
    if None in multipliers:
        raise ValueError("None is not allowed in multipliers tuple. Baseline testing is always included automatically.")
    
    # Ensure all multipliers are positive numbers
    for mult in multipliers:
        if not isinstance(mult, (int, float)) or mult <= 0:
            raise ValueError(f"All multipliers must be positive numbers, got: {mult}")
    
    _log(f"Starting benchmark for {accession} with seed {seed} and multipliers {multipliers}...")
    
    # Define paths
    data_dir = "2_poc_simulacra"
    combined_path = os.path.join(data_dir, f"{accession}_embeddings_with_target.csv")
    max_mult = max(multipliers) if multipliers else 1
    benchmark_cache_path = os.path.join(data_dir, f"{accession}_benchmark_seed_{seed}_mult_{max_mult}x.pkl")
    
    # Check if benchmark results already exist
    if os.path.exists(benchmark_cache_path):
        _log(f"Loading existing benchmark results from {benchmark_cache_path}...")
        with open(benchmark_cache_path, 'rb') as f:
            results = pickle.load(f)
        _log("Benchmark results loaded successfully")
        return results
    
    # Load combined dataframe
    _log("Loading combined embeddings with target...")
    df = pd.read_csv(combined_path, index_col=0)
    _log(f"Loaded dataframe shape: {df.shape}")
    
    # Create train/test split
    _log(f"Creating train/test split (test_fraction={test_fraction})...")
    X = df.drop(columns=['disease'])
    y = df['disease']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_fraction, random_state=seed, stratify=y
    )
    
    _log(f"Split sizes - Train: {len(X_train)}, Test: {len(X_test)}")
    
    # Create training dataframe with target
    train_df = pd.concat([y_train, X_train], axis=1)
    train_df.columns = ['disease'] + list(X_train.columns)
    
    results = {}
    
    # 1. Baseline: Train on original training data only (multiplier=None)
    _log("=== Training Baseline Classifier (no augmentation) ===")
    baseline_model, baseline_metrics = train_ridge_classifier_fair(
        X_train, y_train, X_test, y_test, random_state=seed
    )
    results['baseline'] = {
        'model': baseline_model,
        'metrics': baseline_metrics,
        'train_size': len(X_train),
        'multiplier': None
    }
    _log(f"Baseline - Accuracy: {baseline_metrics['accuracy']:.4f}, F1: {baseline_metrics['f1_macro']:.4f}")
    
    # 2. Train synthesizers and create augmented datasets
    synthesizers = {
        'GaussianCopula': GaussianCopulaSynthesizer,
        'CTGAN': CTGANSynthesizer,
        'TVAE': TVAESynthesizer
    }
    
    for synth_name, synth_class in synthesizers.items():
        _log(f"=== Training {synth_name} Augmented Classifiers ===")
        step_start = datetime.now()
        
        # Create metadata for synthesizer
        _log(f"Creating metadata for {synth_name}...")
        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(train_df)
        
        # Train synthesizer
        _log(f"Training {synth_name} synthesizer (LONG OPERATION)...")
        synth = synth_class(metadata)
        synth.fit(train_df)
        synth_duration = datetime.now() - step_start
        _log(f"{synth_name} synthesizer training completed in {synth_duration.total_seconds():.2f} seconds")
        
        # Generate maximum synthetic data needed (max multiplier)
        max_multiplier = max(multipliers) if multipliers else 1
        _log(f"Generating {max_multiplier}x synthetic data with {synth_name} (LONG OPERATION)...")
        num_synthetic_max = len(train_df) * max_multiplier
        synthetic_data_all = synth.sample(num_rows=num_synthetic_max)
        synth_gen_duration = datetime.now() - step_start
        _log(f"Synthetic data generation completed in {synth_gen_duration.total_seconds():.2f} seconds")
        
        # Test each multiplier
        for multiplier in multipliers:
            _log(f"Testing {synth_name} with {multiplier}x augmentation...")
            
            # Take only the amount needed for this multiplier
            num_synthetic_needed = len(train_df) * multiplier  # multiplier = how many times more synthetic data
            synthetic_data_subset = synthetic_data_all.iloc[:num_synthetic_needed]
            
            # Combine real and synthetic data
            augmented_train = pd.concat([train_df, synthetic_data_subset], axis=0, ignore_index=True)
            total_size = len(train_df) + len(synthetic_data_subset)
            _log(f"Augmented dataset size: {total_size} samples ({len(train_df)} original + {len(synthetic_data_subset)} synthetic)")
            
            # Separate features and target for augmented data
            X_augmented = augmented_train.drop(columns=['disease'])
            y_augmented = augmented_train['disease']
            
            # Train classifier on augmented data
            _log(f"Training classifier on {multiplier}x augmented data...")
            augmented_model, augmented_metrics = train_ridge_classifier_fair(
                X_augmented, y_augmented, X_test, y_test, random_state=seed
            )
            
            _log(f"{synth_name} {multiplier}x - Accuracy: {augmented_metrics['accuracy']:.4f}, F1: {augmented_metrics['f1_macro']:.4f}")
            
            results[f"{synth_name}_{multiplier}x"] = {
                'model': augmented_model,
                'metrics': augmented_metrics,
                'train_size': len(X_augmented),
                'multiplier': multiplier,
                'synthesizer': synth
            }
        
        total_step_duration = datetime.now() - step_start
        _log(f"Total {synth_name} processing time: {total_step_duration.total_seconds():.2f} seconds")
    
    # Save results
    _log(f"Saving benchmark results to {benchmark_cache_path}...")
    with open(benchmark_cache_path, 'wb') as f:
        pickle.dump(results, f)
    _log("Benchmark results saved successfully")
    
    return results
    
def run_benchmark_experiment(accession: str, seeds: list = [42, 931782, 8481962], multipliers = (1, 2)):
    """
    Run benchmark experiment with multiple seeds and augmentation multipliers.
    
    Args:
        accession (str): GSE accession number
        seeds (list): List of random seeds to use
        multipliers: Either a tuple of augmentation multipliers or a single number
                    Single numbers will be cast to tuple. None is not allowed in multipliers.
                    Examples: (1, 2, 5) or 2 (becomes (2,))
    
    Returns:
        dict: Summary statistics across all seeds
    """
    # Handle single number multipliers
    if isinstance(multipliers, (int, float)):
        multipliers = (multipliers,)
    
    # Validate multipliers - None is not allowed
    if None in multipliers:
        raise ValueError("None is not allowed in multipliers tuple. Baseline testing is always included automatically.")
    
    # Ensure all multipliers are positive numbers
    for mult in multipliers:
        if not isinstance(mult, (int, float)) or mult <= 0:
            raise ValueError(f"All multipliers must be positive numbers, got: {mult}")
    
    _log(f"Starting benchmark experiment for {accession} with seeds: {seeds} and multipliers: {multipliers}")
    
    all_results = {}
    
    # Run benchmark for each seed
    for seed in seeds:
        _log(f"\n=== Running benchmark with seed {seed} ===")
        results = benchmark_classifiers(accession, test_fraction=0.2, seed=seed)
        all_results[seed] = results
    
    # Compute statistics across seeds
    _log("\n=== Computing Statistics Across Seeds ===")
    
    # Collect metrics for each method across all seeds
    methods = ['baseline']  # Always include baseline
    for synth_name in ['GaussianCopula', 'CTGAN', 'TVAE']:
        for multiplier in multipliers:
            methods.append(f"{synth_name}_{multiplier}x")
    
    summary_stats = {}
    
    for method in methods:
        accuracies = [all_results[seed][method]['metrics']['accuracy'] for seed in seeds]
        f1_scores = [all_results[seed][method]['metrics']['f1_macro'] for seed in seeds]
        
        summary_stats[method] = {
            'accuracy': {
                'mean': np.mean(accuracies),
                'std': np.std(accuracies),
                'values': accuracies
            },
            'f1_macro': {
                'mean': np.mean(f1_scores),
                'std': np.std(f1_scores),
                'values': f1_scores
            }
        }
    
    # Print summary
    _log("\n=== BENCHMARK RESULTS SUMMARY ===")
    _log("Method                Accuracy (mean ± std)    F1 Macro (mean ± std)")
    _log("-" * 70)
    
    for method in methods:
        acc_mean = summary_stats[method]['accuracy']['mean']
        acc_std = summary_stats[method]['accuracy']['std']
        f1_mean = summary_stats[method]['f1_macro']['mean']
        f1_std = summary_stats[method]['f1_macro']['std']
        
        _log(f"{method:<20} {acc_mean:.4f} ± {acc_std:.4f}        {f1_mean:.4f} ± {f1_std:.4f}")
    
    return all_results, summary_stats

# Execute the benchmark experiment with multiple multipliers
all_results, summary_stats = run_benchmark_experiment('GSE42861', seeds=[42, 931782, 8481962], multipliers=(1, 2, 5))

# Add CSV functionality

def save_results_to_csv(all_results: dict, summary_stats: dict, accession: str, target_column: str = "disease", csv_path: str = None):
    """Save benchmark results to CSV file with comprehensive information."""
    _log("Saving results to CSV file...")
    
    # Define dataset paths
    data_dir = "2_poc_simulacra"
    dnam_path = os.path.join(data_dir, "dnam.csv")
    metadata_path = os.path.join(data_dir, "metadata.csv")
    
    _log(f"Dataset paths - DNAm: {dnam_path}, Metadata: {metadata_path}")
    _log(f"Target column: {target_column}")
    
    # Prepare CSV data
    csv_data = []
    seeds = list(all_results.keys())
    
    for seed in seeds:
        for method, results in all_results[seed].items():
            metrics = results['metrics']
            
            csv_data.append({
                'accession': accession,
                'dnam_path': dnam_path,
                'metadata_path': metadata_path,
                'target_column': target_column,
                'seed': seed,
                'method': method,
                'accuracy': metrics['accuracy'],
                'f1_macro': metrics['f1_macro'],
                'test_size': metrics['test_size'],
                'train_size': results['train_size'],
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
    
    # Add summary statistics rows
    for method in summary_stats.keys():
        csv_data.append({
            'accession': accession,
            'dnam_path': dnam_path,
            'metadata_path': metadata_path,
            'target_column': target_column,
            'seed': 'summary',
            'method': method,
            'accuracy': summary_stats[method]['accuracy']['mean'],
            'f1_macro': summary_stats[method]['f1_macro']['mean'],
            'test_size': 'N/A',
            'train_size': 'N/A',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        
        csv_data.append({
            'accession': accession,
            'dnam_path': dnam_path,
            'metadata_path': metadata_path,
            'target_column': target_column,
            'seed': 'summary',
            'method': f"{method}_std",
            'accuracy': summary_stats[method]['accuracy']['std'],
            'f1_macro': summary_stats[method]['f1_macro']['std'],
            'test_size': 'N/A',
            'train_size': 'N/A',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
    
    # Create DataFrame and save
    results_df = pd.DataFrame(csv_data)
    
    if csv_path is None:
        data_dir = "2_poc_simulacra"
        csv_path = os.path.join(data_dir, f"{accession}_benchmark_results.csv")
    
    results_df.to_csv(csv_path, index=False)
    _log(f"Results saved to {csv_path}")
    _log(f"CSV contains {len(results_df)} rows")
    
    return csv_path

# Save results to CSV
csv_path = save_results_to_csv(all_results, summary_stats, 'GSE42861', target_column='disease')

_log(f"CSV results saved to: {csv_path}")

# Display first few rows of CSV
results_df = pd.read_csv(csv_path)
_log(f"\nCSV Preview (first 10 rows):")
display(results_df)


[2025-10-13 23:39:45] Starting benchmark experiment for GSE42861 with seeds: [42, 931782, 8481962] and multipliers: (1, 2, 5)
[2025-10-13 23:39:45] 
=== Running benchmark with seed 42 ===
[2025-10-13 23:39:45] Starting benchmark for GSE42861 with seed 42 and multipliers (1, 2)...
[2025-10-13 23:39:45] Loading combined embeddings with target...
[2025-10-13 23:39:45] Loaded dataframe shape: (689, 513)
[2025-10-13 23:39:45] Creating train/test split (test_fraction=0.2)...
[2025-10-13 23:39:45] Split sizes - Train: 551, Test: 138
[2025-10-13 23:39:45] === Training Baseline Classifier (no augmentation) ===
[2025-10-13 23:39:45] Baseline - Accuracy: 0.7899, F1: 0.7893
[2025-10-13 23:39:45] === Training GaussianCopula Augmented Classifiers ===
[2025-10-13 23:39:45] Creating metadata for GaussianCopula...
[2025-10-13 23:39:45] Training GaussianCopula synthesizer (LONG OPERATION)...


C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-13 23:40:21] GaussianCopula synthesizer training completed in 36.20 seconds
[2025-10-13 23:40:21] Generating 2x synthetic data with GaussianCopula (LONG OPERATION)...
[2025-10-13 23:40:25] Synthetic data generation completed in 40.34 seconds
[2025-10-13 23:40:25] Testing GaussianCopula with 1x augmentation...
[2025-10-13 23:40:25] Augmented dataset size: 1102 samples (551 original + 551 synthetic)
[2025-10-13 23:40:25] Training classifier on 1x augmented data...
[2025-10-13 23:40:25] GaussianCopula 1x - Accuracy: 0.8116, F1: 0.8110
[2025-10-13 23:40:25] Testing GaussianCopula with 2x augmentation...
[2025-10-13 23:40:25] Augmented dataset size: 1653 samples (551 original + 1102 synthetic)
[2025-10-13 23:40:25] Training classifier on 2x augmented data...
[2025-10-13 23:40:25] GaussianCopula 2x - Accuracy: 0.8116, F1: 0.8110
[2025-10-13 23:40:25] Total GaussianCopula processing time: 40.53 seconds
[2025-10-13 23:40:25] === Training CTGAN Augmented Classifiers ===
[2025-10-13 23:

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

...

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-14 01:49:05] TVAE synthesizer training completed in 1424.56 seconds
[2025-10-14 01:49:05] Generating 2x synthetic data with TVAE (LONG OPERATION)...
[2025-10-14 01:49:10] Synthetic data generation completed in 1429.78 seconds
[2025-10-14 01:49:10] Testing TVAE with 1x augmentation...
[2025-10-14 01:49:10] Augmented dataset size: 1102 samples (551 original + 551 synthetic)
[2025-10-14 01:49:10] Training classifier on 1x augmented data...
[2025-10-14 01:49:10] TVAE 1x - Accuracy: 0.7391, F1: 0.7383
[2025-10-14 01:49:10] Testing TVAE with 2x augmentation...
[2025-10-14 01:49:10] Augmented dataset size: 1653 samples (551 original + 1102 synthetic)
[2025-10-14 01:49:10] Training classifier on 2x augmented data...
[2025-10-14 01:49:10] TVAE 2x - Accuracy: 0.7536, F1: 0.7532
[2025-10-14 01:49:10] Total TVAE processing time: 1429.99 seconds
[2025-10-14 01:49:10] Saving benchmark results to 2_poc_simulacra\GSE42861_benchmark_seed_8481962_mult_2x.pkl...
[2025-10-14 01:49:12] Benchmark r

KeyError: 'GaussianCopula_5x'

NameError: name 'summary_stats' is not defined